# PyHEOM: Brownian Oscillator (CUDA test)

**Before running:** select *Runtime > Change runtime type* and set
*Hardware accelerator* to **GPU** (T4 or later).

Tests the CUDA wheel with a 2-level Brownian-oscillator model.
HEOM (solid) should show underdamped bath oscillation in the coherences;
Redfield (dashed) is Markovian and should not.

In [ ]:
# Install CUDA wheel from Google Drive.
# Upload the .whl file to the root of My Drive (not in a subfolder), then run this cell.
import os, subprocess, sys, glob
from google.colab import drive
drive.mount('/content/drive')

wheels = sorted(glob.glob('/content/drive/MyDrive/*pyheom*cuda12*.whl'))
if not wheels:
    raise FileNotFoundError('No pyheom CUDA 12 wheel found in MyDrive root')
print('Installing:', wheels[-1])
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       '--no-deps', '--no-cache-dir', wheels[-1]])

print('Restarting runtime ...')
os.kill(os.getpid(), 9)

In [ ]:
import importlib.metadata
import pyheom.pylibheom as lb

print('pyheom version:', importlib.metadata.version('pyheom'))
print('eigen backend: ', lb.eigen_is_supported())
print('cuda backend:  ', lb.cuda_is_supported())

if not lb.cuda_is_supported():
    raise RuntimeError(
        'CUDA backend not available. '
        'Check that the runtime type is GPU and the CUDA wheel is installed.'
    )

In [ ]:
import numpy as np
from pyheom import HEOMSolver, RedfieldSolver, Brown, noise_decomposition

# --- parameters (toy values, dimensionless) ---
lambda_0 = 0.1
omega_0  = 1.0
zeta     = 0.5
T        = 1.0
J        = 0.1
truncation_depth  = 10

# --- Hamiltonian ---
omega_1 = np.sqrt(omega_0**2 - zeta**2 * 0.25)
H = np.array([[omega_1, J], [J, 0.0]], dtype=np.complex128)

# --- coupling operator ---
V = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.complex128)

# --- bath ---
J_sd = Brown(lambda_0, zeta, omega_0)
corr = noise_decomposition(J_sd, T=T, type_ltc='psd', n_psd=1, type_psd='n-1/n')
corr.V = V

# --- initial state ---
rho_0 = np.zeros((2, 2), dtype=np.complex128)
rho_0[0, 0] = 1.0

# --- observables ---
P0     = np.array([[1, 0], [0, 0]], dtype=np.complex128)
P1     = np.array([[0, 0], [0, 1]], dtype=np.complex128)
coh_op = np.array([[0, 0], [1, 0]], dtype=np.complex128)  # Tr(coh_op @ rho) = rho_01

# --- time grid ---
t_list = np.arange(0.0, 80.0, 0.1)

print(f'omega_1 = {omega_1:.6f}  truncation_depth = {truncation_depth}')

In [ ]:
from tqdm.auto import tqdm
import time

qme_heom = HEOMSolver(
    H, [corr],
    space='Liouville', format='dense', engine='CUDA', device=0,
    liouville_order='C', solver='rkdp',
    truncation_depth=truncation_depth,
)

with tqdm(total=len(t_list), desc='HEOM') as bar:
    def _cb(t):
        bar.update(1)
    t0 = time.perf_counter()
    result_heom = qme_heom.solve(
        rho_0, t_list, dt=1e-2, atol=1e-8, rtol=1e-6,
        e_ops=[P0, P1, coh_op],
        callback=_cb,
    )
    print(f'HEOM elapsed: {time.perf_counter()-t0:.1f} s')

print(f'rho_00(0) = {result_heom.expect[0][0].real:.4f}  '
      f'rho_11(0) = {result_heom.expect[1][0].real:.4f}  '
      f'Tr = {result_heom.expect[0][0].real + result_heom.expect[1][0].real:.4f}')

In [ ]:
qme_rf = RedfieldSolver(
    H, [corr],
    space='Liouville', format='dense', engine='CUDA', device=0,
    liouville_order='C', solver='lsrk4',
)

t0 = time.perf_counter()
result_rf = qme_rf.solve(
    rho_0, t_list, dt=2.5e-2,
    e_ops=[P0, P1, coh_op],
)
print(f'Redfield elapsed: {time.perf_counter()-t0:.1f} s')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax = axes[0]
ax.plot(result_heom.times, result_heom.expect[0].real,
        label='HEOM rho_00', color='tab:blue')
ax.plot(result_heom.times, result_heom.expect[1].real,
        label='HEOM rho_11', color='tab:orange')
ax.plot(result_rf.times, result_rf.expect[0].real, '--',
        label='Redfield rho_00', color='tab:blue', alpha=0.6)
ax.plot(result_rf.times, result_rf.expect[1].real, '--',
        label='Redfield rho_11', color='tab:orange', alpha=0.6)
ax.set_ylabel('Population')
ax.legend()
ax.set_title('Brownian oscillator (CUDA test)')

ax = axes[1]
coh_h = result_heom.expect[2]
coh_r = result_rf.expect[2]
ax.plot(result_heom.times, coh_h.real,
        label='HEOM Re(rho_01)', color='tab:green')
ax.plot(result_heom.times, coh_h.imag,
        label='HEOM Im(rho_01)', color='tab:red')
ax.plot(result_rf.times, coh_r.real, '--',
        label='Redfield Re(rho_01)', color='tab:green', alpha=0.6)
ax.plot(result_rf.times, coh_r.imag, '--',
        label='Redfield Im(rho_01)', color='tab:red', alpha=0.6)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('t')
ax.set_ylabel('Coherence')
ax.legend()

plt.tight_layout()
plt.show()